<h1 align='center'> 영상처리 프로그래밍 실습 10</h1>

<h6 align='right'> 2025. 6. 5. </h6>

<div class="alert alert-block alert-info">
    
- 파일 이름에서 00000000을 자신의 학번으로, name을 자신의 이름으로 수정하세요.

- 다음 줄에 자신의 이름, 학번, 학과(전공)을 적으세요.

* 이름:   이한석&nbsp;&nbsp;          학번:    20215226&nbsp;&nbsp;         학과(전공): 빅데이터학과
    
</div>

- JupyterLab 문서의 최신 버전은 [JupyterLab Documentation](https://jupyterlab.readthedocs.io/en/stable/index.html#/)을  참고하라

- Markdown은 [Markdown Guide](https://www.markdownguide.org/)를 참고하라.
- [Markdown Cheat Sheet](https://www.markdownguide.org/cheat-sheet/)
- 실험에 사용할 영상은 [영상 데이터베이스](https://drive.google.com/drive/folders/1zbjtkf9nHy9VniuLI4wHilbrN_JBvhYi)에서 다운로드할 것


* 제출 마감: 6월 11일 (수) 오후 10:00까지 최종본을 SmartLEAD제출


In [1]:
import cv2
import matplotlib.pyplot as plt

import numpy as np
print("OpenCV version", cv2.__version__)
print("NumPy version", np.__version__)

OpenCV version 4.11.0
NumPy version 2.0.2


### 문제 1. 
 
다음 조건을 만족하는 함수 perspective_projection_transform(img) 함수를 작성하라.

- OpenCV의 윈도우을 만들고, 입력 영상을 왼쪽에, 입력 영상의 복사본을 오른쪽에 나란히 표시한다
- 왼쪽 영상의 4 각형의 네 점의 위치를 다음 순서대로 마우스로 클릭하면 4 각형을 그린다.
- 왼쪽 위 꼭지점에서 반시계방향으로 네꼭지점을 순서대로 클릭한다.
- 사각형이 완성되면 perspective projection transform된 영상을 오른쪽에 표시한다.
- perspective projection transform 출력 영상의 크기는 입력 영상의 크기와 같다

다음 그림은 'social-media_640.jpg' 파일을 이 프로그램에 적용한 결과 예이다.

![perspective_Tx_Sosical_media](perspective_Tx_Sosical_media.png)

In [ ]:
points = []
img_original = None
img_display = None
img_result = None

def mouse_callback(event, x, y, flags, param):
    global points, img_display, img_original, img_result

    if event == cv2.EVENT_LBUTTONDOWN:
        if len(points) < 4:
            points.append((x, y))
            # 클릭한 점을 표시
            cv2.circle(img_display, (x, y), 5, (0, 255, 0), -1)
            cv2.imshow("Perspective Transform", img_display)

            if len(points) == 4:
                # 4개의 점을 모두 클릭했을 때 사각형 그리기
                for i in range(4):
                    p1 = points[i]
                    p2 = points[(i + 1) % 4]
                    cv2.line(img_display, p1, p2, (0, 0, 255), 2)
                cv2.imshow("Perspective Transform", img_display)

                # Perspective Transform 적용
                transform_image()

def transform_image():
    global points, img_original, img_result, img_display

    if len(points) == 4:
        # 원본 이미지의 크기
        h, w = img_original.shape[:2]

        # 원본 이미지의 4개 점 (사용자가 클릭한 점)
        pts1 = np.float32(points)

        pts2 = np.float32([[0, 0], [0, h], [w, h], [w, 0]]) # (top-left), (bottom-left), (bottom-right), (top-right)

        # 투시 변환 행렬 계산
        M = cv2.getPerspectiveTransform(pts1, pts2)

        # 투시 변환 적용
        img_result_transformed = cv2.warpPerspective(img_original, M, (w, h))

        img_result = img_result_transformed

        # 두 이미지를 나란히 표시
        display_combined_image()

def display_combined_image():
    global img_original, img_result, img_display

    if img_original is None:
        return

    h_orig, w_orig = img_original.shape[:2]
    h_result, w_result = img_result.shape[:2] if img_result is not None else (h_orig, w_orig) 

    combined_img = np.zeros((max(h_orig, h_result), w_orig + w_result, 3), dtype=np.uint8)
    
    combined_img[0:h_orig, 0:w_orig] = img_display if img_display is not None else img_original

    if img_result is not None:
        combined_img[0:h_result, w_orig:w_orig + w_result] = img_result

    cv2.imshow("Perspective Transform", combined_img)

def perspective_projection_transform(img_path):
    global img_original, img_display, img_result, points

    img_original = cv2.imread(img_path)
    if img_original is None:
        print(f"Error: Could not load image {img_path}")
        return

    img_result = np.zeros_like(img_original)

    img_display = img_original.copy()

    cv2.namedWindow("Perspective Transform")
    cv2.setMouseCallback("Perspective Transform", mouse_callback)

    # 초기 화면 표시
    display_combined_image()

    while True:
        key = cv2.waitKey(1) & 0xFF
        if key == 27:  # ESC 키를 누르면 종료
            break
        elif key == ord('r'): # 'r' 키를 누르면 초기화
            points = []
            img_display = img_original.copy()
            img_result = np.zeros_like(img_original)
            display_combined_image()

    cv2.destroyAllWindows()

perspective_projection_transform('social-media_640.jpg')

### 문제 2.

'beach_640.jpg' 파읽을 읽고 k-Means algorithm을 이용해서 5 개의 색만으로 벡터 양자화한 후에,
원 영상과 벡터 양자화된 영상을 나란히 표시하라.

다음 그림은 실행 결과 예이다.

![beach_cluster_5](beach_cluster_5.png)

In [5]:
import cv2
import numpy as np
from sklearn import cluster
from sklearn.utils import shuffle

image_path = 'beach_640.jpg'
img_original = cv2.imread(image_path)

if img_original is None:
    print("이미지 파일을 불러올수 없습니다.")
else:
    h, w, c = img_original.shape

    img_reshaped = img_original.reshape((h * w, c))
    
    K = 5  

    N = 3000
    if h * w < N: # 이미지의 총 픽셀 수가 N보다 작으면, 전체 픽셀 사용
        N = h * w

    img_sample = shuffle(img_reshaped, random_state=0)[:N]

    #K-Means 클러스터링
    kmeans = cluster.KMeans(n_clusters=K, n_init='auto', random_state=0).fit(img_sample)

    #클러스터 중심(양자화된 색상)이랑 각 픽셀의 클러스터 할당 얻기 
    cluster_centers = kmeans.cluster_centers_.round().clip(0, 255).astype(np.uint8)

    # 원본 이미지의 각 픽셀이 어떤 클러스터(즉, 어떤 대표 색상)에 속하는지 예측
    img_labels = kmeans.predict(img_reshaped)

    #양자화된 이미지 재구성
    img_vq = cluster_centers[img_labels]
    
    img_quantized = img_vq.reshape((h, w, c))

    
    h_orig, w_orig = img_original.shape[:2]
    h_quant, w_quant = img_quantized.shape[:2]

    # 두 이미지를 나란히 놓을 빈 NumPy 배열을 생성합니다.
    combined_img = np.zeros((max(h_orig, h_quant), w_orig + w_quant, 3), dtype=np.uint8)

    # 원본 이미지 왼쪽
    combined_img[0:h_orig, 0:w_orig] = img_original

    # 양자화된 이미지 오른쪽
    combined_img[0:h_quant, w_orig:w_orig + w_quant] = img_quantized

    
    cv2.imshow(f"Original vs K-Means Quantized (K={K})", combined_img)
    cv2.waitKey(0) 
    cv2.destroyAllWindows() 

### 문제 3.

문제 2의 프로그램을 다음과 같이 수정하라.

- OpenCV의 trackBar interface를 이용해서 k-Means의 cluster 개수를 조정하면서 벡터 양자화된 영상을 윈도우에 표시하라. 단, cluster 개수의 최솟값은 1로 설정하고, cluster의 개수가 1인 경우에는 원 영상을 표시하라. 따라서 실제 cluster의 최솟값은 2가 된다.


다음 그림은 실행 결과 예이다.

![beach_cluster_tarckbar](beach_cluster_tarckbar.png)

In [7]:
import cv2
import numpy as np
from sklearn import cluster
from sklearn.utils import shuffle

img_original = None
img_display_combined = None
K_current = 1 

# Trackbar 콜백 함수
def on_trackbar_change(k_value):
    global img_original, img_display_combined, K_current

    if k_value < 1: #1보다 작으면 1
        K_current = 1
        cv2.setTrackbarPos("K", "K-Means Quantization", 1)
    else:
        K_current = k_value

    if img_original is None:
        print("Image not loaded.")
        return

    h, w, c = img_original.shape
    img_reshaped = img_original.reshape((h * w, c))

    img_quantized = None

    if K_current == 1:
        # K가 1일 때는 원본 이미지 
        img_quantized = img_original.copy()
    else:
        # K-Means 클러스터링
        N = 3000
        if h * w < N:
            N = h * w
        img_sample = shuffle(img_reshaped, random_state=0)[:N]

        
        kmeans = cluster.KMeans(n_clusters=K_current, n_init='auto', random_state=0).fit(img_sample)

        cluster_centers = kmeans.cluster_centers_.round().clip(0, 255).astype(np.uint8)
        img_labels = kmeans.predict(img_reshaped)
        img_vq = cluster_centers[img_labels]
        img_quantized = img_vq.reshape((h, w, c))


    h_orig, w_orig = img_original.shape[:2]
    h_quant, w_quant = img_quantized.shape[:2]

    combined_img = np.zeros((max(h_orig, h_quant), w_orig + w_quant, 3), dtype=np.uint8)
    combined_img[0:h_orig, 0:w_orig] = img_original
    combined_img[0:h_quant, w_orig:w_orig + w_quant] = img_quantized

    img_display_combined = combined_img
    cv2.imshow("K-Means Quantization", img_display_combined)


if __name__ == "__main__":
    image_file = 'beach_640.jpg' 
    img_original = cv2.imread(image_file)

    if img_original is None:
        print(f"Error: Could not load image {image_file}. Please check the file path.")
    else:
        cv2.namedWindow("K-Means Quantization")
        
        cv2.createTrackbar("K", "K-Means Quantization", 1, 5, on_trackbar_change)

        on_trackbar_change(1)

        while True:
            key = cv2.waitKey(1) & 0xFF
            if key == 27: 
                break

        cv2.destroyAllWindows()

In [22]:
from PyQt5.QtWidgets import (
    QApplication, QMainWindow, QLabel, QSlider,
    QWidget, QHBoxLayout, QVBoxLayout, QSizePolicy,
    QAction, QFileDialog, QMessageBox
)
from PyQt5.QtCore import Qt, QPointF
from PyQt5.QtGui import QPixmap, QImage, QFont, QPainter, QPen
import numpy as np
import cv2
import os
import sys
from sklearn import cluster
from sklearn.utils import shuffle

class MovingAverageViewer(QMainWindow):
    def __init__(self, filename=None):
        super().__init__()
        self.setWindowTitle("PyQt 이미지 처리")
        self.setGeometry(200, 200, 1200, 600) #윈도우 초기 위치와 크기

        self.original_image = None #원본 이미지
        self.filtered_image = None #필터링, 에지 검출 결과 이미지
        self.current_image = None #현재 작업중인 이미지
        self.display_image_for_transform = None  #투시 변환 시 원본 이미지 위에 점을 그리기 위한 이미지
        self.result_image_transformed = None #투시 변환 결과 이미지
        self.quantized_image = None #K-Means 양자화 결과 이미지
        self.rotation_angle = 0 #이미지 회전 각도
        self.perspective_points = [] #K-Means 양자화 모드 활성 여부
        self.k_means_active = False #현재 적용된 필터 종류
        
        self.orig_label = QLabel()
        self.orig_label.setSizePolicy(QSizePolicy.Ignored, QSizePolicy.Ignored)
        self.orig_label.setAlignment(Qt.AlignCenter)
        self.orig_label.mousePressEvent = self.get_mouse_click_for_perspective

        self.filtered_label = QLabel()
        self.filtered_label.setSizePolicy(QSizePolicy.Ignored, QSizePolicy.Ignored)
        self.filtered_label.setAlignment(Qt.AlignCenter)

        #슬라이더
        self.kernel_label = QLabel("Kernel Size / K : 1") 
        self.kernel_label.setFont(QFont("Arial", 12))

        self.slider = QSlider(Qt.Horizontal)
        self.slider.setMinimum(1) #슬라이더 최소값
        self.slider.setMaximum(64) #슬라이더 최대값
        self.slider.setValue(1) #슬라이더 초기값
        self.slider.valueChanged.connect(self.on_slider_change) 

        self.create_menu() #메뉴바 생성

        slider_layout = QHBoxLayout()
        slider_layout.addWidget(QLabel("Value : "))
        slider_layout.addWidget(self.slider)
        slider_layout.addWidget(self.kernel_label)

        image_layout = QHBoxLayout()
        image_layout.addWidget(self.orig_label)
        image_layout.addWidget(self.filtered_label)

        main_layout = QVBoxLayout()
        main_layout.addLayout(image_layout)
        main_layout.addLayout(slider_layout)

        central_widget = QWidget()
        central_widget.setLayout(main_layout)
        self.setCentralWidget(central_widget)

        if filename:
            self.load_image(filename)
        else:
            self.show_message("이미지를 추가하세요")

    def create_menu(self):
        """메뉴바와 메뉴 항목들을 생성합니다."""
        menubar = self.menuBar()

        file_menu = menubar.addMenu("파일")
        open_action = QAction("이미지 열기", self)
        open_action.triggered.connect(self.open_file_dialog)
        file_menu.addAction(open_action)

        delete_action = QAction("이미지 삭제", self)
        delete_action.triggered.connect(self.clear_image)
        file_menu.addAction(delete_action)

        filter_menu = menubar.addMenu("필터")
        filter_menu.addAction(QAction("이동 평균 필터", self, triggered=lambda: self.set_filter_mode("mean")))
        filter_menu.addAction(QAction("가우시안 필터", self, triggered=lambda: self.set_filter_mode("gaussian")))
        filter_menu.addAction(QAction("샤프닝 필터", self, triggered=lambda: self.set_filter_mode("sharpen")))
        filter_menu.addAction(QAction("미디언 필터", self, triggered=lambda: self.set_filter_mode("median")))
        filter_menu.addAction(QAction("박스 필터", self, triggered=lambda: self.set_filter_mode("box")))
    
        edge_menu = menubar.addMenu("에지 검출")
        edge_menu.addAction(QAction("Roberts", self, triggered=lambda: self.set_filter_mode("roberts")))
        edge_menu.addAction(QAction("Prewitt", self, triggered=lambda: self.set_filter_mode("prewitt")))
        edge_menu.addAction(QAction("Sobel", self, triggered=lambda: self.set_filter_mode("sobel")))
        edge_menu.addAction(QAction("Laplacian", self, triggered=lambda: self.set_filter_mode("laplacian")))
        
        transform_menu = menubar.addMenu("변환")
        perspective_transform_action = QAction("투시 변환", self)
        perspective_transform_action.triggered.connect(self.start_perspective_transform)
        transform_menu.addAction(perspective_transform_action)
        reset_perspective_action = QAction("투시 변환 초기화", self)
        reset_perspective_action.triggered.connect(self.reset_perspective_transform)
        transform_menu.addAction(reset_perspective_action)

        quantization_menu = menubar.addMenu("양자화")
        kmeans_quantization_action = QAction("K-Means 양자화", self)
        kmeans_quantization_action.triggered.connect(self.start_k_means_quantization)
        quantization_menu.addAction(kmeans_quantization_action)
        reset_quantization_action = QAction("양자화 초기화", self)
        reset_quantization_action.triggered.connect(self.reset_quantization)
        quantization_menu.addAction(reset_quantization_action)
        
        extra_menu = menubar.addMenu("추가 기능")
        extra_menu.addAction(QAction("가우시안 잡음 추가", self, triggered=self.add_gaussian_noise))
        extra_menu.addAction(QAction("영상 90도 회전", self, triggered=self.rotate_image))

    def show_message(self, text): #메세지 표시
        self.orig_label.setText(text)
        self.orig_label.setFont(QFont("Arial", 16))
        self.orig_label.setAlignment(Qt.AlignCenter)
        self.filtered_label.setText(text)
        self.filtered_label.setFont(QFont("Arial", 16))
        self.filtered_label.setAlignment(Qt.AlignCenter)

    def open_file_dialog(self): #이미지 로드하기
        filename, _ = QFileDialog.getOpenFileName(self, "이미지 열기", "", "Image Files (*.png *.jpg *.bmp)")
        if filename:
            self.load_image(filename)
            self.rotation_angle = 0
            self.current_image = self.original_image.copy()
            self.set_filter_mode("mean") 
            self.slider.setValue(3) 
            self.reset_perspective_transform() 
            self.reset_quantization() 

    def clear_image(self): #초기상태로 되돌리기
        self.original_image = None
        self.current_image = None
        self.filtered_image = None
        self.rotation_angle = 0
        self.quantized_image = None
        self.k_means_active = False 
        self.show_message("이미지를 추가하세요")
        self.reset_perspective_transform()

    def load_image(self, filename): 
        if not os.path.exists(filename):
            QMessageBox.critical(self, "오류", f"파일 '{filename}'이 존재하지 않습니다.")
            return

        image = cv2.imread(filename)
        if image is None:
            QMessageBox.critical(self, "오류", f"이미지를 불러올 수 없습니다 : {filename}")
            return

        self.original_image = image
        self.current_image = self.original_image.copy()
        self.display_image_for_transform = self.original_image.copy()
        self.result_image_transformed = np.zeros_like(self.original_image)
        self.quantized_image = None 
        self.k_means_active = False 
        self.rotation_angle = 0
        self.set_filter_mode("mean") 
        self.slider.setValue(3)
        self.update_original_display()
        self.update_filtered_display()

    def add_gaussian_noise(self): #가우시안 잡음
        if self.current_image is None:
            QMessageBox.warning(self, "경고", "먼저 이미지를 로드하세요.")
            return
        image = self.current_image.copy().astype(np.float32) / 255.0
        mean = 0
        sigma = 0.05
        gauss = np.random.normal(mean, sigma, image.shape)
        noisy = np.clip(image + gauss, 0, 1)
        self.current_image = (noisy * 255).astype(np.uint8)
        self.update_original_display() 
        self.set_filter_mode("mean") 
        self.on_slider_change(self.slider.value())

    def rotate_image(self): #이미지 시계방향으로 90도 회전
        if self.current_image is None:
            QMessageBox.warning(self, "경고", "먼저 이미지를 로드하세요.")
            return
        self.rotation_angle = (self.rotation_angle + 90) % 360
        if self.rotation_angle == 90:
            rotated = cv2.rotate(self.original_image, cv2.ROTATE_90_CLOCKWISE)
        elif self.rotation_angle == 180:
            rotated = cv2.rotate(self.original_image, cv2.ROTATE_180)
        elif self.rotation_angle == 270:
            rotated = cv2.rotate(self.original_image, cv2.ROTATE_90_COUNTERCLOCKWISE)
        else: # 0 or 360
            rotated = self.original_image.copy()
        self.current_image = rotated
        self.update_original_display() 
        self.set_filter_mode("mean") 
        self.on_slider_change(self.slider.value()) 

    
    def on_slider_change(self, value): #슬라이더 값이 변경될 때 호출
        if self.k_means_active:
            self.apply_k_means_quantization(value)
        else:
            self.apply_filter(value)

    def set_filter_mode(self, filter_type):
        self.k_means_active = False
        self.current_filter_type = filter_type
        
        self.slider.setMaximum(51)
        if self.slider.value() % 2 == 0:
            self.slider.setValue(self.slider.value() - 1 if self.slider.value() > 1 else 1)
        self.kernel_label.setText(f"Kernel Size: {self.slider.value()}")
        self.apply_filter(self.slider.value())
        self.update_filtered_display() 

    def apply_filter(self, kernel_size):
        if self.current_image is None:
            return

        if self.current_filter_type in ["median", "gaussian"] and kernel_size % 2 == 0:
            kernel_size = kernel_size - 1 if kernel_size > 1 else 1
            self.slider.setValue(kernel_size) 

        self.kernel_label.setText(f"Kernel Size: {kernel_size}")
        img_to_filter = self.current_image.copy()

        filtered = img_to_filter 

        if self.current_filter_type == "mean":
            filtered = cv2.blur(img_to_filter, (kernel_size, kernel_size))
        elif self.current_filter_type == "gaussian":
            filtered = cv2.GaussianBlur(img_to_filter, (kernel_size, kernel_size), 0)
        elif self.current_filter_type == "median":
            filtered = cv2.medianBlur(img_to_filter, kernel_size)
        elif self.current_filter_type == "box":
            filtered = cv2.boxFilter(img_to_filter, -1, (kernel_size, kernel_size))
        elif self.current_filter_type == "sharpen":
            filtered = self.sharpen_image(img_to_filter)
        elif self.current_filter_type == "roberts":
            filtered = self.roberts_edge(img_to_filter)
        elif self.current_filter_type == "prewitt":
            filtered = self.prewitt_edge(img_to_filter)
        elif self.current_filter_type == "sobel":
            filtered = self.sobel_edge(img_to_filter)
        elif self.current_filter_type == "laplacian":
            filtered = self.laplacian_edge(img_to_filter)

        self.filtered_image = filtered
        self.update_filtered_display()


    def sharpen_image(self, img): #샤프닝 필터
        kernel = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]])
        return cv2.filter2D(img, -1, kernel)

    def roberts_edge(self, img): #로버트 에지 검출 필터
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        kernelx = np.array([[1, 0], [0, -1]])
        kernely = np.array([[0, 1], [-1, 0]])
        edge = cv2.convertScaleAbs(cv2.filter2D(gray, -1, kernelx) + cv2.filter2D(gray, -1, kernely))
        return cv2.cvtColor(edge, cv2.COLOR_GRAY2BGR)

    def prewitt_edge(self, img): #프리윗 에지 검출 필터
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        kernelx = np.array([[1, 0, -1], [1, 0, -1], [1, 0, -1]])
        kernely = np.array([[1, 1, 1], [0, 0, 0], [-1, -1, -1]])
        edge = cv2.convertScaleAbs(cv2.filter2D(gray, -1, kernelx) + cv2.filter2D(gray, -1, kernely))
        return cv2.cvtColor(edge, cv2.COLOR_GRAY2BGR)

    def sobel_edge(self, img): #소벨 에지 검출 필터
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        x = cv2.Sobel(gray, cv2.CV_16S, 1, 0)
        y = cv2.Sobel(gray, cv2.CV_16S, 0, 1)
        edge = cv2.addWeighted(cv2.convertScaleAbs(x), 0.5, cv2.convertScaleAbs(y), 0.5, 0)
        return cv2.cvtColor(edge, cv2.COLOR_GRAY2BGR)

    def laplacian_edge(self, img): #라플라시안 에지 검출 필터
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        edge = cv2.convertScaleAbs(cv2.Laplacian(gray, cv2.CV_16S, ksize=3))
        return cv2.cvtColor(edge, cv2.COLOR_GRAY2BGR)

    def resizeEvent(self, event): #윈도우 크기 변경 시
        super().resizeEvent(event)
        if self.original_image is not None:
            self.on_slider_change(self.slider.value()) 

    def cv2_to_pixmap(self, img): #OpenCV 이미지를 QPixmap로 변환
        if img is None or img.size == 0: 
            return QPixmap()
        rgb_image = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w, ch = rgb_image.shape
        bytes_per_line = ch * w
        qimage = QImage(rgb_image.data, w, h, bytes_per_line, QImage.Format_RGB888)
        return QPixmap.fromImage(qimage)

    def update_original_display(self): #원본 이미지 업데이트
        if self.original_image is None:
            return

        display_img = self.display_image_for_transform.copy()

        for p in self.perspective_points:
            cv2.circle(display_img, p, 5, (0, 255, 0), -1)

    
        if len(self.perspective_points) == 4:
            for i in range(4):
                p1 = self.perspective_points[i]
                p2 = self.perspective_points[(i + 1) % 4]
                cv2.line(display_img, p1, p2, (0, 0, 255), 2)

        self.orig_label.setPixmap(self.cv2_to_pixmap(display_img).scaled(
            self.orig_label.size(), Qt.KeepAspectRatio, Qt.SmoothTransformation))

    def update_filtered_display(self):
        if self.original_image is None:
            return

        display_image = None
        if self.k_means_active and self.quantized_image is not None:
            display_image = self.quantized_image
        elif self.result_image_transformed is not None and np.any(self.result_image_transformed):
            display_image = self.result_image_transformed
        elif self.filtered_image is not None:
            display_image = self.filtered_image
        else:
            display_image = np.zeros_like(self.original_image) 

        self.filtered_label.setPixmap(self.cv2_to_pixmap(display_image).scaled(
            self.filtered_label.size(), Qt.KeepAspectRatio, Qt.SmoothTransformation))


    
    def start_perspective_transform(self): #투시 변환 관련 함수
        if self.original_image is None:
            QMessageBox.warning(self, "경고", "투시 변환을 시작하려면 먼저 이미지를 로드하세요.")
            return

        self.k_means_active = False
        self.quantized_image = None 
        self.slider.setMaximum(51) 
        self.kernel_label.setText(f"Kernel Size: {self.slider.value()}")

        QMessageBox.information(self, "투시 변환", "원본 이미지에 4개의 점을 클릭하여 사각형을 지정하세요.")
        self.perspective_points = []
        self.display_image_for_transform = self.original_image.copy()
        self.result_image_transformed = np.zeros_like(self.original_image)
        self.update_original_display()
        self.update_filtered_display()

    def get_mouse_click_for_perspective(self, event): #4개의 점을 받아 투시 변환 수행
        if self.original_image is None or self.k_means_active: 
            return

        
        if len(self.perspective_points) < 4:
            if event.button() == Qt.LeftButton:
                
                x, y = event.x(), event.y()
                img_w, img_h = self.original_image.shape[1], self.original_image.shape[0]
                label_w, label_h = self.orig_label.width(), self.orig_label.height()

               
                pixmap_ratio = min(label_w / img_w, label_h / img_h)
                displayed_w = int(img_w * pixmap_ratio)
                displayed_h = int(img_h * pixmap_ratio)

                offset_x = (label_w - displayed_w) // 2
                offset_y = (label_h - displayed_h) // 2

    
                orig_x = int((x - offset_x) / pixmap_ratio)
                orig_y = int((y - offset_y) / pixmap_ratio)

                
                if 0 <= orig_x < img_w and 0 <= orig_y < img_h:
                    self.perspective_points.append((orig_x, orig_y))
                    self.update_original_display()

                    if len(self.perspective_points) == 4:
                        self.apply_perspective_transform()
        else: 
            pass

    def apply_perspective_transform(self): #선택된 4개의 점으로 투시변환 적용
        if len(self.perspective_points) != 4:
            return

        h, w = self.original_image.shape[:2]

        pts1 = np.float32(self.perspective_points)
        s = pts1.sum(axis=1)
        diff = np.diff(pts1, axis=1)

        tl = pts1[np.argmin(s)]
        br = pts1[np.argmax(s)]
        tr = pts1[np.argmin(diff)]
        bl = pts1[np.argmax(diff)]

        pts1_sorted = np.float32([tl, tr, br, bl])

        pts2 = np.float32([[0, 0], [w - 1, 0], [w - 1, h - 1], [0, h - 1]])


        M = cv2.getPerspectiveTransform(pts1_sorted, pts2) 
        self.result_image_transformed = cv2.warpPerspective(self.original_image, M, (w, h))
        self.update_filtered_display()
        QMessageBox.information(self, "투시 변환 완료", "투시 변환이 적용되었습니다.")

    def reset_perspective_transform(self): #투시변환 관련 설정을 초기화
        self.perspective_points = []
        if self.original_image is not None:
            self.display_image_for_transform = self.original_image.copy()
            self.result_image_transformed = np.zeros_like(self.original_image)
            self.update_original_display()
            self.update_filtered_display()
        else:
            self.show_message("이미지를 추가하세요")

    
    def start_k_means_quantization(self): #K-Means 양자화 
        if self.original_image is None:
            QMessageBox.warning(self, "경고", "K-Means 양자화를 시작하려면 먼저 이미지를 로드하세요.")
            return

        self.k_means_active = True
        self.perspective_points = [] 
        self.result_image_transformed = np.zeros_like(self.original_image) 
        self.update_original_display() 
        self.slider.setMaximum(64) 
        self.kernel_label.setText(f"K: {self.slider.value()}")
        self.apply_k_means_quantization(self.slider.value()) 

    def apply_k_means_quantization(self, k_value): #K-Means 알고리즘을 사용하여 이미지 색상을 양자화
        if self.original_image is None:
            return

        K_current = max(1, k_value) 
        self.kernel_label.setText(f"K: {K_current}")

        h, w, c = self.original_image.shape
        img_reshaped = self.original_image.reshape((h * w, c))

        if K_current == 1:
            self.quantized_image = self.original_image.copy()
        else:
            N = 3000 
            if h * w < N:
                N = h * w
            img_sample = shuffle(img_reshaped, random_state=0)[:N]

            
            kmeans = cluster.KMeans(n_clusters=K_current, n_init='auto', random_state=0).fit(img_sample)

            cluster_centers = kmeans.cluster_centers_.round().clip(0, 255).astype(np.uint8)
            img_labels = kmeans.predict(img_reshaped)
            img_vq = cluster_centers[img_labels]
            self.quantized_image = img_vq.reshape((h, w, c))

        self.update_filtered_display()

    def reset_quantization(self): #K-Means 양자화 초기화
        self.k_means_active = False
        self.quantized_image = None
        if self.original_image is not None:
            self.slider.setMaximum(51)
            self.slider.setValue(3)
            self.kernel_label.setText(f"Kernel Size: {self.slider.value()}")
            self.set_filter_mode("mean") 
            self.update_filtered_display() 
        else:
            self.show_message("이미지를 추가하세요")


def main():
    app = QApplication.instance()
    if app is None:
        app = QApplication(sys.argv)
    viewer = MovingAverageViewer()
    viewer.show()
    sys.exit(app.exec_())

if __name__ == "__main__":
    main()

SystemExit: 0

c:\Users\larva\AppData\Local\Programs\Python\Python39\lib\site-packages\IPython\core\interactiveshell.py:3558: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
